In [46]:
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
sys.path.append('/Users/kevinlaventure/python_code')
from python_module.pricing_model import SABRModel
from scipy.optimize import minimize, LinearConstraint

# Configure pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:_.2f}')

In [47]:
def compute_option_surface(
    F: float, 
    K_list: list, 
    T_list: list, 
    alpha_list: list, 
    beta: float, 
    rho: float, 
    nu: float,
    r: float, 
    slide_scenario=None,
    slide_type: str = 'spot_only', 
    slide_compute: str = 'option_pnl',
    compute_bs_greeks: bool = True, 
    compute_model_greek: bool = False
) -> pd.DataFrame:
    """
    Computes SABR option prices and Greeks over a grid of strikes and maturities.
    
    Args:
        F: Forward price
        K_list: List of strike prices
        T_list: List of times to maturity (in years)
        alpha_list: List of alpha (volatility) parameters
        beta, rho, nu: SABR parameters
        r: Risk-free rate
        slide_scenario: List of spot bumps (optional)
        slide_type: 'spot_vol' or 'spot_only'
        slide_compute: PnL calculation type ('delta_hedged_pnl', 'option_pnl', 'delta_pnl')
        compute_bs_greeks: If True, returns Black-Scholes Greeks
        compute_model_greek: If True, returns SABR model Greeks
        
    Returns:
        DataFrame with rows for each (K, T, alpha) combination containing:
        - Input parameters: F, K, T, alpha, beta, rho, nu, r, option_type
        - IV: Implied volatility
        - price: Option price
        - Greeks: delta, gamma, vega, theta, vanna, volga
        - Model Greeks (if compute_model_greek=True): sabr_delta, sabr_gamma, sabr_vega, sabr_vanna, sabr_volga, sabr_theta
        - Slides: PnL or price differences for each slide scenario
        
    Note:
        Option type is determined automatically: call if K > F, put if K ≤ F
    """
    results = []
    
    for i in range(len(alpha_list)):
        alpha = alpha_list[i]
        T = T_list[i]
        for K in K_list:
            # Determine option type: call if K > F, put otherwise
            option_type = 'call' if K > F else 'put'
            
            result = SABRModel.compute_option(
                F=F, 
                K=K, 
                T=T, 
                alpha=alpha, 
                beta=beta, 
                rho=rho, 
                nu=nu,
                r=r, 
                option_type=option_type, 
                slide_scenario=slide_scenario,
                slide_type=slide_type, 
                slide_compute=slide_compute,
                compute_bs_greeks=compute_bs_greeks, 
                compute_model_greek=compute_model_greek)
                
            # Build row with inputs and results
            row = {
                'F': F,
                'K': K,
                'T': T,
                'alpha': alpha,
                'beta': beta,
                'rho': rho,
                'nu': nu,
                'r': r,
                'option_type': option_type,
            }
            
            # Add all result fields
            row.update(result)
            
            results.append(row)
    
    # Convert to DataFrame
    df = pd.DataFrame(results)
    
    # Reorder columns: inputs first, then IV and price, then greeks, then slides
    input_cols = ['F', 'K', 'T', 'alpha', 'beta', 'rho', 'nu', 'r', 'option_type']
    price_cols = ['IV', 'price']
    greek_cols = ['delta', 'gamma', 'vega', 'theta', 'vanna', 'volga']
    sabr_greek_cols = ['sabr_delta', 'sabr_gamma', 'sabr_vega', 'sabr_vanna', 'sabr_volga', 'sabr_theta']
    
    # Build column order
    col_order = input_cols + price_cols
    col_order += [c for c in greek_cols if c in df.columns]
    col_order += [c for c in sabr_greek_cols if c in df.columns]
    
    # Add slide columns (remaining columns)
    slide_cols = [c for c in df.columns if c not in col_order]
    col_order += slide_cols
    
    # Reorder dataframe
    df = df[col_order]
    
    return df

In [56]:
# Generate surface with slides
F = 100.0  # Forward price
K_max = 100
K_min = 50
K_step = 1
K_list = np.arange(K_min, K_max + K_step, K_step) # 9 strikes from 80 to 120 --- IGNORE ---
T_list = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]  # Times to maturity
alpha_list = [0.1, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2]  # Alpha (volatility) parameters

beta = 1
rho = -0.9
nu = 0.0001
r = 0.0

# Compute option surface with slides
slide_scenario = [-0.3, -0.1, -0.05, -0.04, -0.03, -0.02, -0.01, 0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.1]
df_surface = compute_option_surface(
    F=F,
    K_list=K_list,
    T_list=T_list,
    alpha_list=alpha_list,
    beta=beta,
    rho=rho,
    nu=nu,
    r=r,
    slide_scenario=slide_scenario,
    compute_bs_greeks=True,
    compute_model_greek=False
)
df_surface['symbol'] = df_surface['K'].astype(str) + '_' + df_surface['T'].astype(str)
df_surface.set_index('symbol', inplace=True)
df =  df_surface.loc[:, slide_scenario]

In [57]:
df_surface.head()

,F,K,T,alpha,beta,rho,nu,r,option_type,IV,price,delta,gamma,vega,theta,vanna,volga,-0.30,-0.10,-0.05,-0.04,-0.03,-0.02,-0.01,0,0.01,0.02,0.03,0.04,0.05,0.10
symbol,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
50_0.1,100.00,50,0.10,0.10,1,-0.90,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
51_0.1,100.00,51,0.10,0.10,1,-0.90,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
52_0.1,100.00,52,0.10,0.10,1,-0.90,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
53_0.1,100.00,53,0.10,0.10,1,-0.90,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00
54_0.1,100.00,54,0.10,0.10,1,-0.90,0.00,0.00,put,0.10,0.00,0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00


In [58]:
df.tail()

,-0.30,-0.10,-0.05,-0.04,-0.03,-0.02,-0.01,0,0.01,0.02,0.03,0.04,0.05,0.10
symbol,,,,,,,,,,,,,,
96_1.0,20.43,4.81,2.13,1.66,1.21,0.78,0.37,-0.02,-0.39,-0.74,-1.07,-1.39,-1.69,-2.96
97_1.0,20.91,5.01,2.24,1.75,1.27,0.82,0.39,-0.02,-0.41,-0.78,-1.13,-1.47,-1.79,-3.13
98_1.0,21.39,5.21,2.34,1.83,1.34,0.86,0.41,-0.02,-0.43,-0.82,-1.19,-1.54,-1.88,-3.32
99_1.0,21.84,5.41,2.44,1.91,1.40,0.90,0.43,-0.02,-0.45,-0.86,-1.25,-1.62,-1.98,-3.50
100_1.0,22.28,5.61,2.54,1.99,1.46,0.94,0.45,-0.02,-0.47,-0.90,-1.31,-1.70,-2.08,-3.69


In [59]:
df.shape

(510, 14)

In [60]:
def compute_hedges(selection_short_scaled, df, short_leg_symbol, q_short):
    res_dict = dict()
    for long_leg_symbol in df.index:
        if long_leg_symbol == short_leg_symbol:
            continue
        selection_long = df.loc[long_leg_symbol]

        pos = selection_long > 0
        neg = selection_long < 0
        zero = selection_long == 0

        # Feasibility check on zero-slope indices
        if not (selection_short_scaled[zero] > 0).all():
            continue
            raise ValueError("Infeasible: selection <= 0 where selection_long == 0")

        L = (-selection_short_scaled[pos] / selection_long[pos]).max() if pos.any() else -np.inf
        U = (-selection_short_scaled[neg] / selection_long[neg]).min() if neg.any() else  np.inf

        if L >= U:
            continue
            raise ValueError("Infeasible: lower bound exceeds upper bound")

        q_long = L + 1e-9 # or any small epsilon
        selection_long_scaled = selection_long * q_long
        selection_combined_scaled = selection_short_scaled + selection_long_scaled 
        res_dict[(short_leg_symbol, long_leg_symbol)] = {'q_short': q_short, 'q_long': q_long, **selection_combined_scaled.to_dict()}
    return res_dict

In [67]:
res = dict()
for short_leg_symbol in df.index:

    selection_short = df.loc[short_leg_symbol]
    q_short = -30_000_000 / selection_short[-0.3]
    selection_short_scaled = selection_short * q_short

    res_temp = compute_hedges(selection_short_scaled, df, short_leg_symbol, q_short)
    res.update(res_temp)

In [68]:
res_df = pd.DataFrame(res).T
res_df = res_df.reset_index(names=['short_symbol', 'long_symbol'])

In [69]:
res_df

,short_symbol,long_symbol,q_short,q_long,-0.30,-0.10,-0.05,-0.04,-0.03,-0.02,-0.01,0,0.01,0.02,0.03,0.04,0.05,0.10
0,50_0.2,68_0.1,-523_345_109_009_513_600.00,63_476_047_844_831.03,12_767_040_414_291.53,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
1,50_0.2,69_0.1,-523_345_109_009_513_600.00,205_814_893_521.84,92_620_499_246.19,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
2,50_0.2,70_0.1,-523_345_109_009_513_600.00,906_914_438.84,754_809_702.69,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
3,51_0.2,68_0.1,-46_860_374_295_079_352.00,696_663_062_552_579.62,140_121_270_500_422.59,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
4,51_0.2,69_0.1,-46_860_374_295_079_352.00,2_258_862_026_041.09,1_016_828_842_719.13,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4946,99_1.0,99_0.9,-1_373_566.07,1_364_828.45,704_834.53,125_906.52,28_430.95,16_995.26,8_434.32,2_770.84,0.00,91.00,2_988.74,8_615.86,16_874.87,27_650.45,40_811.84,137_048.04
4947,100_1.0,100_0.6,-1_346_512.21,1_287_720.39,2_276_314.88,608_660.99,145_551.21,87_851.16,44_038.18,14_649.83,0.00,182.39,15_078.96,44_373.04,87_566.07,143_997.45,212_866.54,708_920.70
4948,100_1.0,100_0.7,-1_346_512.21,1_302_687.36,1_724_598.95,410_884.81,96_397.55,57_963.77,28_921.07,9_548.99,0.00,302.90,10_368.38,29_996.58,58_886.27,96_645.21,142_801.58,478_975.48
4949,100_1.0,100_0.8,-1_346_512.21,1_317_329.15,1_149_246.38,248_461.46,57_426.27,34_417.86,17_099.51,5_604.57,0.00,288.66,6_412.46,18_256.35,35_652.96,58_387.82,86_204.85,290_381.25


In [ ]:
df_surface['ask_price'] = df_surface['vega'] + df_surface['price']
df_surface['bid_price'] = -df_surface['vega'] + df_surface['price']
res_df['bid_price'] = res_df['short_symbol'].map(df_surface['bid_price'])
res_df['ask_price'] = res_df['long_symbol'].map(df_surface['ask_price'])
res_df['e_pnl'] = (res_df['q_short'] * res_df['bid_price'] * -1) - (res_df['q_long'] * res_df['ask_price'])
res_df = res_df[res_df['e_pnl']>0].sort_values('e_pnl', ascending=False).reset_index(drop=True)
res_df['ratio'] = -res_df['q_short'] / res_df['q_long']
res_df = res_df.loc[res_df['ratio']<2]

In [64]:
import numpy as np
import pandas as pd

def compute_all_hedges_fast(df, target_notional=-30_000_000, key_col=-0.3, eps=1e-9):
    """
    Vectorized computation of all (short_leg, long_leg) hedge pairs.
    
    Returns a DataFrame indexed by (short_leg_symbol, long_leg_symbol) with
    columns ['q_short', 'q_long', *df.columns].
    """
    # --- Setup: work with raw numpy for speed ---
    symbols = df.index.to_numpy()
    cols = df.columns.to_numpy()
    M = df.to_numpy(dtype=np.float64)          # shape (N, K)
    N, K = M.shape

    # Locate the key column once
    key_idx = df.columns.get_loc(key_col)

    # --- Step 1: compute q_short for every row ---
    # q_short[i] = target_notional / M[i, key_idx]
    q_short = target_notional / M[:, key_idx]              # shape (N,)

    # Scaled short rows: S[i, k] = M[i, k] * q_short[i]
    S = M * q_short[:, None]                                # shape (N, K)

    # --- Step 2: for each (i = short, j = long) compute L_ij and U_ij ---
    # We need, over k:
    #   pos_mask:  M[j, k] > 0   -> ratio = -S[i, k] / M[j, k];  L = max over these
    #   neg_mask:  M[j, k] < 0   -> ratio = -S[i, k] / M[j, k];  U = min over these
    #   zero_mask: M[j, k] == 0  -> require S[i, k] > 0 (feasibility)
    #
    # Build the full ratio tensor R[i, j, k] = -S[i, k] / M[j, k]
    # Then mask by sign of M[j, k] and reduce over k.

    # Sign masks on M (per long-leg j, per column k)
    pos_jk  = M > 0                                          # (N, K)
    neg_jk  = M < 0                                          # (N, K)
    zero_jk = ~(pos_jk | neg_jk)                             # (N, K)

    # Safe denominator to avoid divide-by-zero warnings; we'll mask it out
    safe_M = np.where(M == 0, 1.0, M)                        # (N, K)

    # Ratio tensor: shape (N_short, N_long, K)
    # R[i, j, k] = -S[i, k] / safe_M[j, k]
    # Use broadcasting: -S[:, None, :] / safe_M[None, :, :]
    R = -S[:, None, :] / safe_M[None, :, :]                  # (N, N, K)

    # For L: take max over k where pos_jk[j, k]; default -inf
    R_for_L = np.where(pos_jk[None, :, :], R, -np.inf)       # (N, N, K)
    L = R_for_L.max(axis=2)                                  # (N, N)

    # For U: take min over k where neg_jk[j, k]; default +inf
    R_for_U = np.where(neg_jk[None, :, :], R, np.inf)        # (N, N, K)
    U = R_for_U.min(axis=2)                                  # (N, N)

    # --- Step 3: feasibility on zero-slope columns ---
    # For each (i, j): require S[i, k] > 0 for all k where zero_jk[j, k]
    # Equivalent to: min over k in zero_jk of S[i, k] > 0  (treat empty as +inf)
    S_for_zero = np.where(zero_jk[None, :, :], S[:, None, :], np.inf)   # (N, N, K)
    zero_min   = S_for_zero.min(axis=2)                                  # (N, N)
    zero_ok    = zero_min > 0                                            # (N, N)

    # If ANY (i, j) violates zero feasibility, raise (matches original behavior).
    # Original code raises only when iterating that pair, but the check is the same.
    if not zero_ok.all():
        bad_i, bad_j = np.where(~zero_ok)
        raise ValueError(
            f"Infeasible: selection <= 0 where selection_long == 0 "
            f"for short={symbols[bad_i[0]]}, long={symbols[bad_j[0]]}"
        )

    # --- Step 4: build feasibility mask and q_long ---
    feasible = L < U                                          # (N, N), strict like original
    np.fill_diagonal(feasible, False)                         # skip i == j

    q_long = L + eps                                          # (N, N)

    # --- Step 5: combined scaled selection for every feasible (i, j) ---
    # combined[i, j, k] = S[i, k] + q_long[i, j] * M[j, k]
    # We only need feasible pairs; gather them.
    ii, jj = np.where(feasible)                               # both shape (P,)
    if ii.size == 0:
        return pd.DataFrame(columns=['q_short', 'q_long', *cols])

    combined = S[ii] + q_long[ii, jj][:, None] * M[jj]        # (P, K)

    # --- Step 6: assemble result DataFrame ---
    out = pd.DataFrame(combined, columns=cols)
    out.insert(0, 'q_long',  q_long[ii, jj])
    out.insert(0, 'q_short', q_short[ii])
    out.index = pd.MultiIndex.from_arrays(
        [symbols[ii], symbols[jj]],
        names=['short_leg_symbol', 'long_leg_symbol']
    )
    return out

In [65]:
compute_all_hedges_fast(df, target_notional=-30_000_000, key_col=-0.3, eps=1e-9)

q_short  \
short_leg_symbol long_leg_symbol                               
50_0.2           68_0.1          -523_345_109_009_513_600.00   
                 69_0.1          -523_345_109_009_513_600.00   
                 70_0.1          -523_345_109_009_513_600.00   
51_0.2           68_0.1           -46_860_374_295_079_352.00   
                 69_0.1           -46_860_374_295_079_352.00   
...                                                      ...   
99_1.0           99_0.9                        -1_373_566.07   
100_1.0          100_0.6                       -1_346_512.21   
                 100_0.7                       -1_346_512.21   
                 100_0.8                       -1_346_512.21   
                 100_0.9                       -1_346_512.21   

                                                 q_long  \
short_leg_symbol long_leg_symbol                          
50_0.2           68_0.1           63_476_047_844_831.03   
                 69_0.1              205_814_893_521.84   
                 70_0.1                  906_914_438.84   
51_0.2           68_0.1          696_663_062_552_579.62   
                 69_0.1            2_258_862_026_041.09   
...                                                 ...   
99_1.0           99_0.9                    1_364_828.45   
100_1.0          100_0.6                   1_287_720.39   
                 100_0.7                   1_302_687.36   
                 100_0.8                   1_317_329.15   
                 100_0.9                   1_331_883.57   

                                                  -0.30      -0.10      -0.05  \
short_leg_symbol long_leg_symbol                                                
50_0.2           68_0.1           12_767_040_414_291.53       0.00       0.00   
                 69_0.1               92_620_499_246.19       0.00       0.00   
                 70_0.1                  754_809_702.69       0.00       0.00   
51_0.2           68_0.1          140_121_270_500_422.59       0.00       0.00   
                 69_0.1            1_016_828_842_719.13       0.00       0.00   
...                                                 ...        ...        ...   
99_1.0           99_0.9                      704_834.53 125_906.52  28_430.95   
100_1.0          100_0.6                   2_276_314.88 608_660.99 145_551.21   
                 100_0.7                   1_724_598.95 410_884.81  96_397.55   
                 100_0.8                   1_149_246.38 248_461.46  57_426.27   
                 100_0.9                     569_353.88 113_450.31  25_902.95   

                                     -0.04     -0.03     -0.02  -0.01      0  \
short_leg_symbol long_leg_symbol                                               
50_0.2           68_0.1               0.00      0.00      0.00   0.00   0.00   
                 69_0.1               0.00      0.00      0.00   0.00   0.00   
                 70_0.1               0.00      0.00      0.00   0.00   0.00   
51_0.2           68_0.1               0.00      0.00      0.00   0.00   0.00   
                 69_0.1               0.00      0.00      0.00   0.00   0.00   
...                                    ...       ...       ...    ...    ...   
99_1.0           99_0.9          16_995.26  8_434.32  2_770.84   0.00  91.00   
100_1.0          100_0.6         87_851.16 44_038.18 14_649.83   0.00 182.39   
                 100_0.7         57_963.77 28_921.07  9_548.99   0.00 302.90   
                 100_0.8         34_417.86 17_099.51  5_604.57   0.00 288.66   
                 100_0.9         15_479.01  7_659.08  2_492.03   0.00 179.41   

                                      0.01      0.02      0.03       0.04  \
short_leg_symbol long_leg_symbol                                            
50_0.2           68_0.1               0.00      0.00      0.00       0.00   
                 69_0.1               0.00      0.00      0.00       0.00   
                 70_0.1               0.00      0

In [84]:
import numpy as np
import pandas as pd

def compute_hedges_fast(selection_short_scaled_arr, values, index_arr, short_idx,
                        short_leg_symbol, q_short, columns):
    """
    Vectorized computation of hedges for a single short leg against all long legs.
    
    Parameters
    ----------
    selection_short_scaled_arr : np.ndarray, shape (n_cols,)
    values : np.ndarray, shape (n_rows, n_cols)  -- df.values
    index_arr : np.ndarray of object, shape (n_rows,) -- df.index.values
    short_idx : int -- row index of short leg
    """
    n_rows, n_cols = values.shape

    # Mask out the short leg row itself
    row_mask = np.ones(n_rows, dtype=bool)
    row_mask[short_idx] = False

    long_vals = values[row_mask]                    # (n_rows-1, n_cols)
    long_symbols = index_arr[row_mask]              # (n_rows-1,)

    pos_mask = long_vals > 0                        # (n_rows-1, n_cols)
    neg_mask = long_vals < 0
    zero_mask = ~(pos_mask | neg_mask)              # equivalent to long_vals == 0

    # Feasibility: where long == 0, short_scaled must be > 0
    short_pos = selection_short_scaled_arr > 0      # (n_cols,)
    # For each row, every zero-slope column must have short_pos True
    infeasible_zero = (zero_mask & ~short_pos).any(axis=1)
    if infeasible_zero.any():
        bad = long_symbols[infeasible_zero][0]
        raise ValueError(
            f"Infeasible: selection <= 0 where selection_long == 0 "
            f"(short={short_leg_symbol}, long={bad})"
        )

    # Compute -short_scaled / long_vals safely; we'll mask out invalid entries
    with np.errstate(divide='ignore', invalid='ignore'):
        ratios = -selection_short_scaled_arr[None, :] / long_vals  # (n_rows-1, n_cols)

    # L = max over pos entries; if no pos, -inf
    pos_ratios = np.where(pos_mask, ratios, -np.inf)
    L = pos_ratios.max(axis=1)                      # (n_rows-1,)

    # U = min over neg entries; if no neg, +inf
    neg_ratios = np.where(neg_mask, ratios, np.inf)
    U = neg_ratios.min(axis=1)                      # (n_rows-1,)

    feasible = L < U                                # (n_rows-1,)

    if not feasible.any():
        return {}

    # Filter to feasible rows only
    L_f = L[feasible]
    long_vals_f = long_vals[feasible]
    long_symbols_f = long_symbols[feasible]

    q_long = L_f + 1e-9                             # (k,)
    # selection_long_scaled = long_vals * q_long  (broadcast over columns)
    long_scaled = long_vals_f * q_long[:, None]     # (k, n_cols)
    combined = selection_short_scaled_arr[None, :] + long_scaled  # (k, n_cols)

    # Build result dict
    res = {}
    cols_list = list(columns)
    for i, long_sym in enumerate(long_symbols_f):
        row = combined[i]
        d = {'q_short': q_short, 'q_long': q_long[i]}
        d.update(zip(cols_list, row))
        res[(short_leg_symbol, long_sym)] = d
    return res


def compute_all_hedges_fast(df):
    """
    Drop-in replacement for the outer loop.
    Returns the same `res` dict the original code builds.
    """
    values = np.ascontiguousarray(df.values, dtype=np.float64)
    index_arr = df.index.to_numpy()
    columns = df.columns

    # Locate the column corresponding to -0.3 once
    # (matches df.loc[symbol][-0.3] semantics, i.e. label-based lookup)
    col_neg03_idx = df.columns.get_loc(-0.3)

    res = {}
    for short_idx in range(values.shape[0]):
        short_leg_symbol = index_arr[short_idx]
        selection_short = values[short_idx]

        q_short = -60_000_000 / selection_short[col_neg03_idx]
        selection_short_scaled = selection_short * q_short

        res_temp = compute_hedges_fast(
            selection_short_scaled, values, index_arr,
            short_idx, short_leg_symbol, q_short, columns,
        )
        res.update(res_temp)
    return res


def compute_hedges_for_product(df, target_symbol):
    """
    Compute the best hedges for a single target product (used as the short leg)
    against all other products as long legs.
    """
    values = np.ascontiguousarray(df.values, dtype=np.float64)
    index_arr = df.index.to_numpy()
    columns = df.columns

    col_neg03_idx = df.columns.get_loc(-0.3)

    # Locate the row for the target product
    # Using np.where avoids issues if the index isn't a pandas Index lookup
    matches = np.where(index_arr == target_symbol)[0]
    if len(matches) == 0:
        raise KeyError(f"Symbol {target_symbol!r} not found in df.index")
    short_idx = int(matches[0])

    selection_short = values[short_idx]
    q_short = -60_000_000 / selection_short[col_neg03_idx]
    selection_short_scaled = selection_short * q_short

    return compute_hedges_fast(
        selection_short_scaled, values, index_arr,
        short_idx, target_symbol, q_short, columns,
    )

# Usage:
# res = compute_hedges_for_product(df, "MY_SYMBOL")

# Usage:
# 

In [85]:
res = compute_all_hedges_fast(df)

In [86]:
pd.DataFrame(res).T

q_short                   q_long  \
50_0.2  68_0.1  -1_046_690_218_019_027_200.00   126_952_095_689_662.06   
        69_0.1  -1_046_690_218_019_027_200.00       411_629_787_043.68   
        70_0.1  -1_046_690_218_019_027_200.00         1_813_828_877.69   
51_0.2  68_0.1     -93_720_748_590_158_704.00 1_393_326_125_105_159.25   
        69_0.1     -93_720_748_590_158_704.00     4_517_724_052_082.19   
...                                       ...                      ...   
99_1.0  99_0.9                  -2_747_132.13             2_729_656.90   
100_1.0 100_0.6                 -2_693_024.41             2_575_440.79   
        100_0.7                 -2_693_024.41             2_605_374.73   
        100_0.8                 -2_693_024.41             2_634_658.29   
        100_0.9                 -2_693_024.41             2_663_767.14   

                                 -0.30        -0.10      -0.05      -0.04  \
50_0.2  68_0.1   25_534_080_828_583.07         0.00       0.00       0.00   
        69_0.1      185_240_998_492.39         0.00       0.00       0.00   
        70_0.1        1_509_619_405.39         0.00       0.00       0.00   
51_0.2  68_0.1  280_242_541_000_845.19         0.00       0.00       0.00   
        69_0.1    2_033_657_685_438.27         0.00       0.00       0.00   
...                                ...          ...        ...        ...   
99_1.0  99_0.9            1_409_669.06   251_813.04  56_861.89  33_990.51   
100_1.0 100_0.6           4_552_629.77 1_217_321.99 291_102.41 175_702.32   
        100_0.7           3_449_197.90   821_769.62 192_795.11 115_927.54   
        100_0.8           2_298_492.77   496_922.93 114_852.55  68_835.73   
        100_0.9           1_138_707.75   226_900.63  51_805.89  30_958.02   

                    -0.03     -0.02  -0.01      0      0.01      0.02  \
50_0.2  68_0.1       0.00      0.00   0.00   0.00      0.00      0.00   
        69_0.1       0.00      0.00   0.00   0.00      0.00      0.00   
        70_0.1       0.00      0.00   0.00   0.00      0.00      0.00   
51_0.2  68_0.1       0.00      0.00   0.00   0.00      0.00      0.00   
        69_0.1       0.00      0.00   0.00   0.00      0.00      0.00   
...                   ...       ...    ...    ...       ...       ...   
99_1.0  99_0.9  16_868.65  5_541.67   0.00 181.99  5_977.48 17_231.72   
100_1.0 100_0.6 88_076.37 29_299.65   0.00 364.77 30_157.92 88_746.08   
        100_0.7 57_842.15 19_097.98   0.00 605.79 20_736.75 59_993.17   
        100_0.8 34_199.02 11_209.13   0.00 577.33 12_824.93 36_512.69   
        100_0.9 15_318.16  4_984.07   0.00 358.83  6_004.72 16_836.41   

                      0.03       0.04       0.05         0.10  
50_0.2  68_0.1        0.00       0.00       0.00         0.00  
        69_0.1        0.00       0.00       0.00         0.00  
        70_0.1        0.00       0.00       0.00         0.00  
51_0.2  68_0.1        0.00       0.00       0.00         0.00  
        69_0.1        0.00       0.00       0.00         0.00  
...                    ...        ...        ...          ...  
99_1.0  99_0.9   33_749.73  55_300.90  81_623.68   274_096.08  
100_1.0 100_0.6 175_132.13 287_994.91 425_733.07 1_417_841.40  
        100_0.7 117_772.54 193_290.42 285_603.15   957_950.97  
        100_0.8  71_305.92 116_775.64 172_409.70   580_762.51  
        100_0.9  32_710.85  53_447.27  78_831.37   266_178.35  

[4951 rows x 16 columns]

In [87]:
pd.DataFrame(compute_hedges_for_product(df, '50_0.2')).T

q_short                 q_long  \
50_0.2 68_0.1 -1_046_690_218_019_027_200.00 126_952_095_689_662.06   
       69_0.1 -1_046_690_218_019_027_200.00     411_629_787_043.68   
       70_0.1 -1_046_690_218_019_027_200.00       1_813_828_877.69   

                              -0.30  -0.10  -0.05  -0.04  -0.03  -0.02  -0.01  \
50_0.2 68_0.1 25_534_080_828_583.07   0.00   0.00   0.00   0.00   0.00   0.00   
       69_0.1    185_240_998_492.39   0.00   0.00   0.00   0.00   0.00   0.00   
       70_0.1      1_509_619_405.39   0.00   0.00   0.00   0.00   0.00   0.00   

                 0  0.01  0.02  0.03  0.04  0.05  0.10  
50_0.2 68_0.1 0.00  0.00  0.00  0.00  0.00  0.00  0.00  
       69_0.1 0.00  0.00  0.00  0.00  0.00  0.00  0.00  
       70_0.1 0.00  0.00  0.00  0.00  0.00  0.00  0.00